# BoTTLE × LIBERO: closed-loop latency on a free T4

Stock LIBERO pauses physics while the policy thinks, so success never depends on latency, and BoTTLE's
E3 had to assume it doesn't. This notebook removes that assumption:

1. **measured**: run SmolVLA on the Colab T4 and hold the robot still for the *measured* inference time of every chunk.
2. **emulate**: replay the same rollouts with the latency BoTTLE *predicts* for edge devices (Thor, AGX Orin, Orin Nano) and more.
3. **compare**: measured T4 latency vs BoTTLE's T4 prediction, and measured time per success vs E3's prediction.

Runtime: GPU (T4). ~10 episodes/task × 10 tasks per setting; results are saved to Drive so you can resume.

## 1. Install (≈5–10 min)
Stops with the pip log if anything fails, so later cells never run on a broken install.

In [ ]:
import os, sys, subprocess
os.environ["MUJOCO_GL"] = "egl"
# egl_probe 1.0.2 declares cmake_minimum_required(2.8.12); harmless with cmake<4, needed if CMake 4 is on PATH
os.environ["CMAKE_POLICY_VERSION_MINIMUM"] = "3.5"
LEROBOT_REV = "fe1c54594f6ba275fccf210b9f7b11f0e2324b89"  # wrapper patches lerobot internals; pinned

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-6000:])
        raise RuntimeError(f"failed: {cmd}")
    return r.stdout

sh("apt-get -qq install -y libegl1 libgl1 > /dev/null 2>&1 || true")
sh("rm -rf /content/BoTTLE && git clone -q --depth 1 https://github.com/PKM100/BoTTLE.git /content/BoTTLE")
# egl_probe / hf-egl-probe (LIBERO deps) build with CMake. Colab's `cmake` is pip's Python shim, which
# fails inside pip's isolated build env, so pre-build them without isolation. cmake<4 also satisfies lerobot.
sh(f'{sys.executable} -m pip install -q "cmake>=3.29.0.1,<4" setuptools wheel')
sh(f"{sys.executable} -m pip install --no-build-isolation hf-egl-probe==1.0.2 egl_probe==1.0.2")
sh(f'{sys.executable} -m pip install -q "lerobot[libero,smolvla] @ git+https://github.com/huggingface/lerobot.git@{LEROBOT_REV}"')
sh(f"{sys.executable} -m pip install -q -e /content/BoTTLE")
print(sh(f'{sys.executable} -c "import lerobot, libero, torch, bottlebench; print(\\"lerobot ok | torch\\", torch.__version__, \\"| cuda\\", torch.cuda.is_available())"'))
print(sh("nvidia-smi --query-gpu=name,memory.total --format=csv"))

Optional: persist results across Colab sessions.

In [ ]:
SAVE = "/content/bottle_sim"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE = "/content/drive/MyDrive/bottle_sim"
except Exception as e:
    print("Drive not mounted, saving locally:", e)
os.makedirs(SAVE, exist_ok=True)

## 2. Settings
Latencies are BoTTLE's SmolVLA predictions from E3 (`experiments/results/libero_cost.csv`), plus a sweep.

In [ ]:
POLICY = "lerobot/smolvla_libero"
SUITE = "libero_spatial"
H = 10                 # executed actions per chunk (E3 assumes 10 for SmolVLA)
N_PER_TASK = 10        # 50 is the standard protocol; 10 keeps a setting at ~10-20 min on a T4
TASK_IDS = list(range(10))

PREDICTED_MS = {"rtx-4090": 182, "h100-sxm": 208, "t4": 281, "jetson-agx-thor": 308,
                "jetson-agx-orin-64gb": 493, "jetson-orin-nano-super": 669}
SWEEP_MS = [100, 1000, 1500]
SETTINGS = [("stock", "none", 0), ("t4-measured", "measured", 0)] + \
           [(f"pred-{d}", "emulate", ms) for d, ms in PREDICTED_MS.items() if d != "t4"] + \
           [(f"sweep-{ms}ms", "emulate", ms) for ms in SWEEP_MS]
SETTINGS

## 3. Run (resumable: finished `(setting, task)` pairs are skipped)

In [ ]:
import subprocess, json, pathlib, sys
WRAP = "/content/BoTTLE/experiments/sim/bottle_libero_eval.py"

def run(name, mode, ms, task_id):
    out = pathlib.Path(SAVE) / name / f"task{task_id}"
    if (out / "bottle_timing.json").exists():
        return
    cmd = [sys.executable, WRAP, f"--bottle.mode={mode}", f"--bottle.latency_ms={ms}",
           f"--policy.path={POLICY}", f"--policy.n_action_steps={H}",
           "--env.type=libero", f"--env.task={SUITE}", f"--env.task_ids=[{task_id}]",
           "--eval.batch_size=1", f"--eval.n_episodes={N_PER_TASK}", "--eval.use_async_envs=false",
           f"--output_dir={out}"]
    r = subprocess.run(cmd, capture_output=True, text=True, stdin=subprocess.DEVNULL)
    if r.returncode or not (out / "bottle_timing.json").exists():
        print(r.stdout[-3000:], r.stderr[-6000:])
        raise RuntimeError(f"{name} task {task_id} failed (see log above)")
    print(name, task_id, r.stdout.strip().splitlines()[-1])

for name, mode, ms in SETTINGS:
    for t in TASK_IDS:
        run(name, mode, ms, t)

## 4. Results

In [ ]:
import statistics, math
import pandas as pd

rows, t4_ms = [], []
for name, mode, ms in SETTINGS:
    eps, calls = [], []
    for t in TASK_IDS:
        f = pathlib.Path(SAVE) / name / f"task{t}" / "bottle_timing.json"
        if f.exists():
            d = json.loads(f.read_text()); eps += d["episodes"]; calls += d["infer_ms_all"][1:]
    if not eps:
        continue
    if name == "t4-measured":
        t4_ms = calls
    k, n = sum(e["success"] for e in eps), len(eps)
    pa = (k + 2) / (n + 4)                                  # Agresti-Coull 95% interval
    half = 1.96 * math.sqrt(pa * (1 - pa) / (n + 4))
    lo, hi = max(pa - half, 0), min(pa + half, 1)
    sim_s = sum(e["steps"] for e in eps) / 20.0           # simulated robot time incl. holds
    rows.append({"setting": name, "latency_ms": ms if mode == "emulate" else
                 (statistics.median(calls) if mode == "measured" else 0),
                 "n": n, "success": k / n, "ci95": f"[{lo:.2f}, {hi:.2f}]",
                 "robot_s_per_success": sim_s / k if k else float("inf")})
assert rows, f"no results in {SAVE}: run section 3 first (it stops with the error if a run fails)"
df = pd.DataFrame(rows).sort_values("latency_ms"); df

In [ ]:
pred_t4 = PREDICTED_MS["t4"]
if t4_ms:
    med = statistics.median(t4_ms)
    print(f"T4 SmolVLA chunk latency: measured median {med:.0f} ms (n={len(t4_ms)}), "
          f"BoTTLE predicted {pred_t4} ms -> error {100*(pred_t4-med)/med:+.0f}%")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(df.latency_ms, df.success, "o-"); ax[0].set_xlabel("inference latency (ms)"); ax[0].set_ylabel("success")
for _, r in df.iterrows(): ax[0].annotate(r.setting.replace("pred-", ""), (r.latency_ms, r.success), fontsize=7)
ax[1].plot(df.latency_ms, df.robot_s_per_success, "o-"); ax[1].set_xlabel("inference latency (ms)")
ax[1].set_ylabel("robot seconds / success")
plt.tight_layout(); plt.savefig(f"{SAVE}/success_vs_latency.png", dpi=150)

## 5. Compare with BoTTLE E3
E3 assumes latency only adds time (success fixed at the published rate). If success falls with latency here,
E3's sync numbers for slow devices are optimistic: multiply E3's time and cost per success by
`success(stock) / success(device)`.

In [ ]:
base = df.set_index("setting")
p0 = base.loc["stock", "success"]
for name in base.index:
    if name.startswith("pred-") or name == "t4-measured":
        p = base.loc[name, "success"]
        print(f"{name:28s} success {p:.2f} (stock {p0:.2f}) -> E3 correction x{p0/p if p else float('inf'):.2f}")
df.to_csv(f"{SAVE}/summary.csv", index=False)